In [ ]:
import os
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# หาฟอนต์ไทยที่ติดตั้งอยู่แล้วในระบบ (macOS มี Thonburi built-in)
thai_fonts = ['Thonburi', 'Ayuthaya', 'Krungthep', 'Sarabun', 'Norasi']
available = {f.name for f in fm.fontManager.ttflist}

font_name = next((f for f in thai_fonts if f in available), None)

if font_name:
    plt.rcParams['font.family'] = font_name
    print(f"✓ ใช้ฟอนต์: {font_name}")
else:
    print("⚠️ ไม่พบฟอนต์ไทย — ตัวอักษรไทยอาจแสดงผลไม่ถูกต้อง")

plt.rcParams['axes.unicode_minus'] = False

# โหลดข้อมูลจาก database
conn = sqlite3.connect("results.db")
df = pd.read_sql("SELECT * FROM results", conn)
conn.close()

print(f"ข้อมูลทั้งหมด: {len(df)} แถว")
df.head()

In [85]:
prompt_list = df['prompt'].unique()
prompt_map = {prompt: f"Q{i+1}" for i, prompt in enumerate(prompt_list)}
df['prompt_label'] = df['prompt'].map(prompt_map)

# แสดง legend บอกว่า Q1 = อะไร
print("Prompt Legend:")
for prompt, label in prompt_map.items():
    print(f"{label}: {prompt}")

Prompt Legend:
Q1: แคมเปญ Double Day ซื้อสกินแคร์แบรนด์เนมที่ไหนลดเยอะที่สุด?
Q2: Flash Sale เครื่องสำอางช่วง Mid-month แพลตฟอร์มไหนมีโค้ดส่วนลดคุ้มที่สุด?
Q3: ถ้าซื้อเซรั่มมาแล้วแพ้ แพลตฟอร์มไหนรับคืนสินค้าง่ายที่สุด?
Q4: ช่วง 11.11 หรือ 12.12 ซื้อเครื่องสำอางแพลตฟอร์มไหนถูกกว่ากัน?
Q5: ซื้อเซรั่มวิตามินซี ราคาหลักร้อย แพลตฟอร์มไหนมีให้เลือกเยอะที่สุด?
Q6: ของแถมจากแบรนด์เครื่องสำอาง แพลตฟอร์มไหนให้คุ้มที่สุด?
Q7: ซื้อเครื่องสำอางแบรนด์เนมมั่นใจได้ว่าของแท้ ต้องซื้อที่ไหน?
Q8: ซื้อน้ำหอม Chanel ออนไลน์ แพลตฟอร์มไหนรับประกันของแท้ 100%?
Q9: เจอของปลอมบนแพลตฟอร์มออนไลน์บ่อยไหม แล้วไปซื้อที่ไหนปลอดภัย?
Q10: เครื่องสำอางเกาหลีนำเข้าถูกต้อง แพลตฟอร์มไหนมี อย. ครบ?
Q11: อยากหาแบรนด์เครื่องสำอางเกาหลีใหม่ๆ ที่ยังไม่มีขายในไทย ไปหาที่ไหนดี?
Q12: แบรนด์สกินแคร์ญี่ปุ่นหายาก เช่น SKII, Shiseido แพลตฟอร์มไหนมีให้เลือกเยอะ?
Q13: เครื่องสำอางจากประเทศไทย (indie brand) แพลตฟอร์มไหนรวมไว้เยอะที่สุด?
Q14: ซื้อเครื่องสำอางแบรนด์ดัง + niche brand ในที่เดียว ไปที่ไหนดี?
Q15: สั่งเครื่องสำอางออนไลน์ต้องกา

In [ ]:
visibility = df.groupby('brand')['mentioned'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
colors = ['green' if x > 0 else 'red' for x in visibility.values]
visibility.plot(kind='bar', color=colors)
plt.title('Brand Visibility — Times Mentioned by AI')
plt.xlabel('Brand')
plt.ylabel('Times Mentioned')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('images/01_brand_visibility.png', dpi=150)
plt.show()

In [ ]:
ranked = df[df['rank'] > 0][['brand', 'rank', 'prompt_label']].sort_values('rank')

plt.figure(figsize=(10, 5))
for brand in ranked['brand'].unique():
    data = ranked[ranked['brand'] == brand]
    plt.scatter(data['prompt_label'], data['rank'], label=brand, s=100)

plt.title('Brand Rank by Prompt')
plt.xlabel('Prompt')
plt.ylabel('Rank (lower = better)')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig('images/02_brand_rank_by_prompt.png', dpi=150)
plt.show()

In [ ]:
sentiment_counts = df[df['mentioned'] == True]['sentiment'].value_counts()

plt.figure(figsize=(6, 6))
colors = ['green', 'gray', 'red']
sentiment_counts.plot(kind='pie', autopct='%1.1f%%', colors=colors)
plt.title('Sentiment Breakdown')
plt.ylabel('')
plt.tight_layout()
plt.savefig('images/03_sentiment_breakdown.png', dpi=150)
plt.show()

In [ ]:
category_brand = df.pivot_table(
    index='prompt_label',
    columns='brand',
    values='mentioned',
    aggfunc='sum',
    fill_value=0
)

category_brand = category_brand[category_brand.sum().sort_values(ascending=False).index]
category_brand = category_brand.loc[category_brand.sum(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(12, 8))
sns.heatmap(category_brand, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Brand Mentions by Prompt')
plt.xlabel('Brand')
plt.ylabel('Prompt')
plt.tight_layout()
plt.savefig('images/04_heatmap_brand_by_prompt.png', dpi=150)
plt.show()

In [90]:
# Average rank per brand (ต่ำ = ดี)
avg_rank = df[df['mentioned'] == True].groupby('brand')['rank'].mean().sort_values()

print("Average Rank (ต่ำ = AI แนะนำก่อน):")
print(avg_rank)

Average Rank (ต่ำ = AI แนะนำก่อน):
brand
Sephora      2.076923
Shopee       2.153846
Lazada       2.434783
Konvy        2.888889
EVEANDBOY    2.900000
Watsons      3.000000
Beautrium    3.500000
Boots        5.500000
Name: rank, dtype: float64


In [ ]:
plt.figure(figsize=(10, 5))
avg_rank.plot(kind='barh', color='skyblue')
plt.title('Average Rank by Brand (Lower = Better)')
plt.xlabel('Average Rank')
plt.ylabel('Brand')
plt.axvline(x=3, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('images/05_avg_rank_by_brand.png', dpi=150)
plt.show()

In [ ]:
# % ที่ AI mention brands ในแต่ละ category
category_rate = (
    df.dropna(subset=['prompt_category'])
    .groupby('prompt_category')['mentioned']
    .mean() * 100
)

print("Mention Rate by Category (%):")
print(category_rate.sort_values(ascending=False).map('{:.2f}'.format))

In [ ]:
plt.figure(figsize=(12, 8))
category_rate.sort_values().plot(kind='barh', color='coral')
plt.title('AI Mention Rate by Category (%)', fontsize=16)
plt.xlabel('Mention Rate (%)', fontsize=12)
plt.tight_layout()
plt.savefig('images/06_mention_rate_by_category.png', dpi=150)
plt.show()

## สรุปผลการวิเคราะห์

จากการรวบรวมคำตอบของ AI จากคำถาม 30 ข้อ พบว่า Shopee และ Lazada ยังคงเป็นตัวเลือกหลักที่ AI แนะนำ โดยถูกกล่าวถึงใน **78.8%** และ **69.7%** ของคำตอบทั้งหมด ตามลำดับ ส่วน Sephora ถูกพูดถึงใน **39.4%** โดยเฉพาะในคำถามเกี่ยวกับการซื้อเครื่องสำอางแบรนด์เนมและความน่าเชื่อถือของแพลตฟอร์ม

**สิ่งที่น่าสนใจ**

Allnii แทบไม่ปรากฏในคำตอบของ AI เลย (**0.0%**) ในขณะที่ Konvy ถูกกล่าวถึงเพียง **27.3%** ซึ่งบ่งชี้ว่าทั้ง 2 แบรนด์อาจต้องปรับกลยุทธ์ content marketing เพื่อให้ AI รู้จักและสามารถแนะนำได้

เมื่อพิจารณาจากประเภทคำถาม พบว่า AI มักแนะนำ brands มากที่สุดในคำถามเกี่ยวกับ **การซื้อเครื่องสำอางต่างจังหวัด, ผู้ซื้อครั้งแรก และของแท้แบรนด์เนม (66.7%)** ในขณะที่คำถามเกี่ยวกับ **ของปลอม, customer service และการหาแบรนด์ใหม่** ได้รับการแนะนำน้อยที่สุด (0%) เพราะ AI มักเลี่ยงระบุแพลตฟอร์มเมื่อมีความเสี่ยงด้านความน่าเชื่อถือ

**ลำดับการแนะนำ**

แม้ Shopee จะถูกพูดถึงบ่อยที่สุด แต่เมื่อดูที่ average rank พบว่า **Sephora** มักถูกแนะนำเป็นอันดับต้นๆ (avg rank = **2.08**) ส่วน **Boots** ถึงแม้จะถูกกล่าวถึงบ้าง แต่มักอยู่ในลำดับหลังๆ (avg rank = **5.50**)